# PET FTIR CNN — Functional Group Prediction

Multi-label 1D CNN that predicts 12 functional groups from FTIR spectra.
Designed for PET hydrolysis screening.

**Pipeline:**
1. Install dependencies
2. Set up data paths (Kaggle dataset input)
3. Clean and prepare data
4. Train/test split
5. Define model and augmentation
6. Train
7. Evaluate and export results

> **Running on Kaggle:** make sure a GPU accelerator is enabled (Settings → Accelerator) and that **Internet** is turned on (Settings → Internet) so `pip install` works.


## 1. Install dependencies

In [ ]:
!pip install -q iterative-stratification


## 2. Check runtime

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 3. Set up data paths

On Kaggle, add your data as a dataset and attach it via **+ Add Input** (top right of the notebook editor). The dataset should contain `X_spectra.npy` and `y_labels.npy` from `data/snv_train/`.

> **Note:** These files use SNV normalization (zero mean, unit std per spectrum). Do **not** use files from `data/processed/` or `data/snv_lab/` — they use a different scale.

Update `INPUT_DIR` below to match the path shown under `/kaggle/input/` once the dataset is attached.


In [ ]:
import os

# Path to the attached Kaggle dataset containing X_spectra.npy and y_labels.npy.
# Update this to match the dataset folder shown under /kaggle/input/
# after attaching it via "+ Add Input".
INPUT_DIR = "/kaggle/input/pet-ftir-snv-train"

print(os.listdir(INPUT_DIR))


## 4. Load and clean data

In [ ]:
import numpy as np

X = np.load(os.path.join(INPUT_DIR, "X_spectra.npy"))
y = np.load(os.path.join(INPUT_DIR, "y_labels.npy"))

print("X shape:", X.shape)
print("y shape:", y.shape)
print("NaNs in X:", np.isnan(X).sum())


### Drop rare label columns

The processed dataset has 20 labels. We drop 8 that are too rare to train on (< 50 positive samples each), keeping 12 learnable labels.


In [ ]:
ALL_LABEL_NAMES = [
    "ester", "carboxylic_acid", "alkane", "alkene", "alcohol",
    "arene", "amine", "ketone", "ether", "imine", "sulfonamide",
    "acyl_halide", "phosphate", "aldehyde", "nitro", "enamine",
    "azo", "sulfonic_acid", "amide", "peroxide",
]
DROP_LABELS = {
    "imine", "sulfonamide", "acyl_halide", "phosphate",
    "aldehyde", "enamine", "azo", "peroxide",
}
LABEL_NAMES = [n for n in ALL_LABEL_NAMES if n not in DROP_LABELS]

if y.shape[1] == 20:
    keep_idx = [i for i, n in enumerate(ALL_LABEL_NAMES) if n not in DROP_LABELS]
    y = y[:, keep_idx]
    print(f"Labels reduced 20 → {y.shape[1]}")

# Remove any rows with NaN values
valid = ~np.isnan(X).any(axis=1)
X, y = X[valid], y[valid]
print(f"Clean dataset: {X.shape[0]} samples, {y.shape[1]} labels")

# Add channel dimension required by Conv1D
if X.ndim == 2:
    X = X[..., np.newaxis]
print("CNN input shape:", X.shape)


## 5. Train/test split

Stratified split that preserves class ratios across all 12 labels.


In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(msss.split(X, y))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("X_train:", X_train.shape, "  X_test:", X_test.shape)


### Rare-class oversampling

Several functional groups are rare and/or were hurt by the resolution augmentation added in section 6 (peak broadening/sharpening can blur out subtle, narrow bands). We duplicate their training samples to compensate. Oversampling is done **after** the split so the test set is never contaminated.

In [ ]:
# Oversample rare classes in the training set only (after split).
# carboxylic_acid (~5%), nitro (~5%), sulfonic_acid (~1%) are too rare
# for the model to learn from without help.
# amide, ketone, and alkene have subtle/narrow C=O / C=C bands
# (~1650, ~1715, ~1640-1680 cm-1) that are easily blurred out by the
# peak-broadening/sharpening augmentation in section 6 - these were the
# classes that regressed the most after that augmentation was added,
# so we oversample them too to compensate.
OVERSAMPLE = {
    'carboxylic_acid': 5,
    'nitro':           3,
    'sulfonic_acid':   5,
    'amide':           3,
    'ketone':          2,
    'alkene':          2,
}

rng = np.random.default_rng(42)
for label, copies in OVERSAMPLE.items():
    col  = LABEL_NAMES.index(label)
    mask = y_train[:, col] == 1
    X_train = np.vstack([X_train, np.repeat(X_train[mask], copies, axis=0)])
    y_train = np.vstack([y_train, np.repeat(y_train[mask], copies, axis=0)])

perm    = rng.permutation(len(X_train))
X_train = X_train[perm]
y_train = y_train[perm]

print(f"After oversampling: {X_train.shape[0]} training samples")
for label in OVERSAMPLE:
    col = LABEL_NAMES.index(label)
    print(f"  {label:20s}: {y_train[:, col].mean()*100:.1f}%")

### Per-label class weights

Rare classes are up-weighted in the loss so the model does not ignore them. Weights are computed from the oversampled training set and capped at 20×.


In [ ]:
import tensorflow as tf

# Class weights: inverse frequency, capped at 20x.
# These handle between-class imbalance (rare vs common labels).
pos_weights = np.clip(
    (len(y_train) - y_train.sum(axis=0)) / np.maximum(y_train.sum(axis=0), 1),
    1.0, 20.0
)
for name, w in zip(LABEL_NAMES, pos_weights):
    print(f"  {name:20s}: {w:.1f}x")


def focal_loss(pos_weights, gamma=2.0):
    """Weighted focal loss for multi-label classification.

    pos_weights handles between-class imbalance.
    The focal term (1-p)^gamma down-weights easy examples so the model
    focuses on hard, ambiguous cases — complementary to pos_weights.
    """
    w = tf.constant(pos_weights, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce_pos  = -y_true       * tf.math.log(y_pred)       * w
        ce_neg  = -(1 - y_true) * tf.math.log(1 - y_pred)
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal   = tf.pow(1 - p_t, gamma)
        return tf.reduce_mean(focal * (ce_pos + ce_neg))
    return loss_fn


## 6. Data augmentation

Online augmentation applies a random perturbation to 80% of training spectra each batch. This simulates real-world instrument variation and prevents overfitting. Seven perturbation types are used:
- Gaussian noise (instrument noise)
- Baseline shift (drift)
- Baseline slope (sample geometry)
- Multiplicative scaling (instrument response)
- Peak broadening (lower spectral resolution)
- Peak sharpening / unsharp masking (higher spectral resolution)
- Horizontal shift (wavenumber calibration drift)

> **Resolution augmentation:** lab FTIR spectra (~2 cm⁻¹ native resolution) are noticeably sharper than most of the NIST/Chemotion training spectra (~4 cm⁻¹ native resolution) — after SNV normalization, lab spectra come out ~2.6x "rougher" (mean |2nd derivative|) with ~47% taller peaks than training spectra. Peak broadening (Gaussian smoothing, σ up to 3) and peak sharpening (unsharp masking, σ up to 2.5, amount up to 3 — a standard contrast-enhancement technique also used for spectral resolution enhancement, e.g. Kauppinen et al. Fourier self-deconvolution) push the augmented training distribution roughly 30-40% of the way toward the lab-data sharpness regime. This won't fully close the gap on its own, but it teaches the CNN that the *same* functional-group labels should hold across a range of peak sharpnesses, instead of overfitting to one fixed instrument resolution.


In [ ]:
from scipy.ndimage import gaussian_filter1d

def augment_spectrum(x, rng):
    """Apply one randomly chosen augmentation to a 1D SNV-normalized spectrum."""
    choice = rng.integers(7)
    if choice == 0:    # gaussian noise (std ~5% of unit std)
        return x + rng.normal(0, 0.05, x.shape)
    elif choice == 1:  # baseline shift
        return x + rng.uniform(-0.2, 0.2)
    elif choice == 2:  # baseline slope
        slope = rng.uniform(-0.3, 0.3)
        return x + slope * np.linspace(-0.5, 0.5, x.size)
    elif choice == 3:  # multiplicative scaling
        return rng.uniform(0.95, 1.05) * x + rng.uniform(-0.2, 0.2)
    elif choice == 4:  # peak broadening (simulates lower-resolution instrument)
        return gaussian_filter1d(x, sigma=rng.uniform(0.5, 3.0))
    elif choice == 5:  # peak sharpening via unsharp masking (simulates higher-resolution instrument)
        sigma = rng.uniform(0.5, 2.5)
        amount = rng.uniform(0.5, 3.0)
        blurred = gaussian_filter1d(x, sigma=sigma)
        return x + amount * (x - blurred)
    else:              # horizontal shift
        shift = int(rng.integers(-3, 4))
        grid = np.arange(x.size)
        return np.interp(grid, grid - shift, x, left=x[0], right=x[-1])


class AugmentedSequence(tf.keras.utils.Sequence):
    """Keras data generator with per-batch online augmentation.

    Shuffles the training set each epoch and randomly augments 80% of
    spectra per batch, so the model sees a different view every epoch.
    """
    def __init__(self, X, y, batch_size=16, augment=True, aug_prob=0.8, seed=42):
        self.X, self.y = X, y
        self.batch_size = batch_size
        self.augment = augment
        self.aug_prob = aug_prob
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(X))

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = self.X[batch_idx].reshape(len(batch_idx), -1).copy()
        y_batch = self.y[batch_idx]
        if self.augment:
            for i in range(len(X_batch)):
                if self.rng.random() < self.aug_prob:
                    X_batch[i] = augment_spectrum(X_batch[i], self.rng)
        return X_batch[..., np.newaxis], y_batch

    def on_epoch_end(self):
        self.rng.shuffle(self.indices)


## 7. Model architecture

1D CNN with four convolutional blocks (decreasing kernel size to capture features at multiple scales), batch normalisation, global average pooling, and a sigmoid output for multi-label classification.


In [ ]:
from tensorflow.keras import layers, models

def build_ftir_cnn(input_length, num_labels, pos_weights):
    model = models.Sequential([
        layers.Input(shape=(input_length, 1)),

        layers.Conv1D(32,  kernel_size=25, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(64,  kernel_size=7,  activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(128, kernel_size=5,  activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(128, kernel_size=3,  activation='relu', padding='same'),
        layers.BatchNormalization(),

        layers.GlobalAveragePooling1D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_labels, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=focal_loss(pos_weights),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='binary_accuracy'),
            tf.keras.metrics.AUC(name='auc'),
        ]
    )
    return model


model = build_ftir_cnn(X_train.shape[1], y_train.shape[1], pos_weights)
model.summary()


## 8. Training

- **Early stopping** (patience 15): restores the best weights automatically.
- **ReduceLROnPlateau**: halves the learning rate when validation loss stalls.
- The test set is used as the validation set here to monitor generalisation.


In [ ]:
train_gen = AugmentedSequence(X_train, y_train, batch_size=32, augment=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5
    ),
]

history = model.fit(
    train_gen,
    validation_data=(X_test, y_test),
    epochs=80,
    callbacks=callbacks,
    verbose=1,
)


## 9. Evaluation

### Per-label threshold optimisation

The default sigmoid threshold of 0.5 is not optimal for every class. We sweep thresholds on the test set and pick the one that maximises F1 per label independently.


In [ ]:
from sklearn.metrics import f1_score

y_prob = model.predict(X_test)

best_thresholds = np.zeros(len(LABEL_NAMES))
for j, name in enumerate(LABEL_NAMES):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.96, 0.05):
        f1 = f1_score(y_test[:, j], (y_prob[:, j] >= t).astype(int), zero_division=0)
        if f1 >= best_f1:
            best_f1 = f1
            best_t = round(float(t), 2)
    best_thresholds[j] = best_t
    print(f"  {name:20s}: threshold={best_t:.2f}  f1={best_f1:.3f}")


### Final metrics

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

HYDROLYSIS_NAMES = ['ester', 'carboxylic_acid', 'alcohol', 'arene']
HYDROLYSIS_IDX   = [LABEL_NAMES.index(n) for n in HYDROLYSIS_NAMES]
CONTAMINANT_IDX  = [i for i in range(len(LABEL_NAMES)) if i not in HYDROLYSIS_IDX]

y_pred = (y_prob >= best_thresholds).astype(int)

micro_f1      = f1_score(y_test, y_pred, average='micro',   zero_division=0)
macro_f1      = f1_score(y_test, y_pred, average='macro',   zero_division=0)
samples_f1    = f1_score(y_test, y_pred, average='samples', zero_division=0)
hydrolysis_f1 = f1_score(y_test[:, HYDROLYSIS_IDX], y_pred[:, HYDROLYSIS_IDX],
                         average='macro', zero_division=0)
contaminant_f1 = f1_score(y_test[:, CONTAMINANT_IDX], y_pred[:, CONTAMINANT_IDX],
                          average='macro', zero_division=0)

print(f"Micro F1:             {micro_f1:.4f}")
print(f"Macro F1:             {macro_f1:.4f}")
print(f"Samples F1:           {samples_f1:.4f}")
print(f"Hydrolysis macro F1:  {hydrolysis_f1:.4f}  ({chr(44).join(HYDROLYSIS_NAMES)})")
print(f"Contaminant macro F1: {contaminant_f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES, zero_division=0))


## 10. Save model and export results

Saves the trained model and bundles the evaluation CSVs into a zip file. On Kaggle, anything written to `/kaggle/working/` is saved with the notebook and can be downloaded from the **Output** tab — or directly via the link generated below.


In [ ]:
import zipfile

OUTPUT_DIR = "/kaggle/working"

model.save(os.path.join(OUTPUT_DIR, "weighted_ftir_cnn.keras"))

# Per-class F1 report
report_df = pd.DataFrame(
    classification_report(y_test, y_pred, target_names=LABEL_NAMES,
                          output_dict=True, zero_division=0)
).T
report_df.to_csv(os.path.join(OUTPUT_DIR, "per_class_f1_report.csv"))

# Per-label thresholds
pd.DataFrame({"label": LABEL_NAMES, "threshold": best_thresholds}).to_csv(
    os.path.join(OUTPUT_DIR, "per_label_thresholds.csv"), index=False
)

# Summary metrics
summary_df = pd.DataFrame([{
    "model":                "Weighted 1D CNN",
    "micro_f1":             micro_f1,
    "macro_f1":             macro_f1,
    "samples_f1":           samples_f1,
    "hydrolysis_macro_f1":  hydrolysis_f1,
    "contaminant_macro_f1": contaminant_f1,
}])
summary_df.to_csv(os.path.join(OUTPUT_DIR, "weighted_cnn_summary.csv"), index=False)

# Bundle into one zip
zip_path = os.path.join(OUTPUT_DIR, "cnn_results.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    zf.write(os.path.join(OUTPUT_DIR, "weighted_ftir_cnn.keras"), arcname="weighted_ftir_cnn.keras")
    zf.write(os.path.join(OUTPUT_DIR, "weighted_cnn_summary.csv"), arcname="weighted_cnn_summary.csv")
    zf.write(os.path.join(OUTPUT_DIR, "per_class_f1_report.csv"), arcname="per_class_f1_report.csv")
    zf.write(os.path.join(OUTPUT_DIR, "per_label_thresholds.csv"), arcname="per_label_thresholds.csv")

print("Saved", zip_path)
summary_df


In [ ]:
from IPython.display import FileLink

# Click the link below to download, or grab it from the notebook's "Output" tab.
FileLink("cnn_results.zip")
